# 06 – Associationsregler (ARM)

**Syfte:** Hitta regler av typen

> `kommungrupp=C` & `bnp_indikator=svag` → `fruktsamhet=hög`

och pröva om de håller på år som modellen inte har sett.

| | |
|---|---|
| **Indata** | `data/processed/arm_underlag_lag1.csv` (huvudanalys), `arm_underlag_lag2.csv` (känslighet), från `05` |
| **Resultat** | `resultat/arm_regler_lag1.csv`, `resultat/arm_regler_lag2.csv`, `resultat/arm_regler_lag1_utan_bortfall.csv` |
| **Beroenden** | Kör `05` först. Paketet `mlxtend` behövs. |

### Så läses en regel `X → Y`
- **Support:** andel av alla transaktioner där både X och Y gäller.
- **Confidence:** andel av transaktionerna med X där också Y gäller, alltså sannolikheten för Y givet X.
- **Lift:** confidence delat med andelen Y totalt. Lift > 1 betyder att X gör Y vanligare än genomsnittet.

### Två extra mått för just den här datan
Standardmåtten räcker inte här, av två skäl:

1. **Kommuntypen förklarar mycket i sig.** Om C-kommuner ofta har hög fruktsamhet får *varje* regel som börjar med `kommungrupp=C` hög lift, oavsett konjunktur. Måttet **tillskott** jämför därför regelns confidence med *samma kommungrupp utan konjunkturvillkor*. Tillskottet visar vad konjunkturen tillför utöver kommuntypen, och det är det frågeställningen gäller.
2. **Konjunkturen är nationell.** Ett konjunkturvillkor gäller alla kommuner samma år. En regel kan ha tusentals stödjande kommun-år men ändå bara vila på två olika år. Måttet **antal_ar** räknar hur många olika år som står bakom regeln. Det är det verkliga antalet oberoende observationer av konjunkturen.

## 1. Inställningar och parametrar

- `MIN_SUPPORT`: lägsta support för att en itemmängd ska räknas som frekvent. Gränsen är låg (1 %) eftersom kombinationer av kommungrupp och konjunkturklass är naturligt ovanliga. Det egentliga urvalet görs med de följande villkoren.
- `MIN_AR`: minsta antal *olika år* bakom en regel. Med färre år är regeln en anekdot.
- `MIN_TILLSKOTT`: konjunkturvillkoret måste höja confidence med minst så mycket (i andelar) jämfört med bara kommungruppen.
- `MIN_AR_TEST`: en regel räknas som **stabil** om den har positivt tillskott även på teståren och bygger på minst så många testår.

Cellen kontrollerar också att `mlxtend` finns.

In [1]:
from pathlib import Path

import pandas as pd

# Notebooken körs normalt från mappen Assignment.
# Körs den från projektroten letar vi upp Assignment ändå.
BAS = Path.cwd()
if not (BAS / "data" / "raw").is_dir() and (BAS / "Assignment").is_dir():
    BAS = BAS / "Assignment"

RAW = BAS / "data" / "raw"
PROCESSED = BAS / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

assert RAW.is_dir(), "Hittar inte data/raw. Öppna notebooken från mappen Assignment."

try:
    from mlxtend.frequent_patterns import association_rules, fpgrowth
except ModuleNotFoundError as fel:
    raise ModuleNotFoundError(
        "mlxtend saknas. Installera det i notebookens miljö: pip install mlxtend"
    ) from fel

RESULTAT = BAS / "resultat"
RESULTAT.mkdir(exist_ok=True)

INDIKATORER = ["bnp_indikator", "hushallens_konsumtion", "sysselsattning"]
MIN_SUPPORT = 0.01
MAX_LANGD = 5
MIN_AR = 3
MIN_TILLSKOTT = 0.05
MIN_AR_TEST = 2

## 2. Hjälpfunktioner

Tre funktioner används både i huvudanalysen och i känslighetsanalysen. Därför definieras de en gång här.

- `till_items`: gör om en rad med klasser till **items** i form av sant/falskt-kolumner, till exempel `kommungrupp=C` och `bnp_indikator=svag`. Alla tänkbara klasser skapas alltid, även om en klass saknas i en viss period. Då har tränings- och testdata exakt samma kolumner.
- `matt`: räknar support, confidence, lift och antal olika år för en regel på valfri datamängd. Samma funktion används på tränings- och testdata, så att måtten beräknas på identiskt sätt.
- `ar_regeltyp`: avgör om en regel besvarar frågeställningen. Kraven är att slutledet är exakt en fruktsamhetsklass och att villkoret innehåller exakt en kommungrupp, minst ett konjunkturvillkor och ingen fruktsamhetsklass.

In [2]:
KATEGORIER = {
    "kommungrupp": ["A", "B", "C"],
    "fruktsamhet": ["låg", "medel", "hög"],
    **{ind: ["svag", "normal", "stark"] for ind in INDIKATORER},
}
KALLKOLUMN = {"kommungrupp": "kommungrupp", "fruktsamhet": "fruktsamhet_klass",
              **{ind: f"{ind}_klass" for ind in INDIKATORER}}


def till_items(df):
    """En sant/falskt-kolumn per item, t.ex. 'kommungrupp=C'."""
    kolumner = {f"{namn}={klass}": df[KALLKOLUMN[namn]].eq(klass)
                for namn, klasser in KATEGORIER.items() for klass in klasser}
    return pd.DataFrame(kolumner, index=df.index)


def matt(items, ar, villkor, slutled):
    """Support, confidence, lift och antal olika år för regeln villkor → slutled."""
    har_villkor = items[list(villkor)].all(axis=1)
    if har_villkor.sum() == 0:
        return {"support": 0.0, "confidence": float("nan"), "lift": float("nan"), "antal_ar": 0}
    traffar = har_villkor & items[slutled]
    confidence = traffar.sum() / har_villkor.sum()
    return {
        "support": traffar.mean(),
        "confidence": confidence,
        "lift": confidence / items[slutled].mean(),
        "antal_ar": ar[har_villkor].nunique(),
    }


def ar_regeltyp(villkor, slutled):
    """Besvarar regeln frågeställningen?"""
    typ = lambda item: item.split("=")[0]
    return (
        len(slutled) == 1 and typ(next(iter(slutled))) == "fruktsamhet"
        and sum(typ(i) == "kommungrupp" for i in villkor) == 1
        and sum(typ(i) in INDIKATORER for i in villkor) >= 1
        and not any(typ(i) == "fruktsamhet" for i in villkor)
    )

## 3. Läs in underlaget och skapa transaktioner

Huvudanalysen använder **1 års fördröjning**. Varje rad (kommun och år) blir en transaktion med fem items: kommungrupp, fruktsamhetsklass och en klass per konjunkturindikator. Transaktionerna delas i träning och test enligt kolumnen `del` från `05`.

In [3]:
underlag = pd.read_csv(PROCESSED / "arm_underlag_lag1.csv", encoding="utf-8-sig", dtype={"Kommunkod": str})

items = till_items(underlag)
assert (items.sum(axis=1) == len(KATEGORIER)).all(), "Varje transaktion ska ha exakt ett item per variabel."

tran = underlag["del"] == "traning"
items_tran, ar_tran = items.loc[tran], underlag.loc[tran, "ar"]
items_test, ar_test = items.loc[~tran], underlag.loc[~tran, "ar"]

print(f"Transaktioner: {len(items_tran)} träning ({ar_tran.min()}–{ar_tran.max()}), "
      f"{len(items_test)} test ({ar_test.min()}–{ar_test.max()})")
display(items.head(3))

Transaktioner: 3670 träning (2003–2015), 2808 test (2016–2025)


,kommungrupp=A,kommungrupp=B,kommungrupp=C,fruktsamhet=låg,fruktsamhet=medel,fruktsamhet=hög,bnp_indikator=svag,bnp_indikator=normal,bnp_indikator=stark,hushallens_konsumtion=svag,hushallens_konsumtion=normal,hushallens_konsumtion=stark,sysselsattning=svag,sysselsattning=normal,sysselsattning=stark
0,True,False,False,False,True,False,False,True,False,False,True,False,False,False,True
1,True,False,False,True,False,False,True,False,False,True,False,False,False,True,False
2,True,False,False,False,True,False,False,True,False,True,False,False,True,False,False


## 4. Baslinje: fruktsamhet per kommungrupp

Innan konjunkturen tas med visar vi hur fruktsamhetsklasserna fördelar sig **inom varje kommungrupp**. Det är baslinjen som varje regel jämförs med. Värdet är confidence för regeln `kommungrupp=X → fruktsamhet=Y`.

Skillnaderna mellan grupperna är i sig ett resultat. De visar strukturella skillnader mellan kommuntyperna oavsett konjunktur.

In [4]:
for del_ in ["traning", "test"]:
    print(f"\nAndel per fruktsamhetsklass inom kommungrupp – {del_}:")
    tabell = pd.crosstab(underlag.loc[underlag["del"] == del_, "kommungrupp"],
                         underlag.loc[underlag["del"] == del_, "fruktsamhet_klass"], normalize="index")
    display(tabell[KATEGORIER["fruktsamhet"]].round(3))


Andel per fruktsamhetsklass inom kommungrupp – traning:


fruktsamhet_klass,låg,medel,hög
kommungrupp,,,
A,0.256,0.381,0.363
B,0.354,0.310,0.336
C,0.353,0.330,0.317



Andel per fruktsamhetsklass inom kommungrupp – test:


fruktsamhet_klass,låg,medel,hög
kommungrupp,,,
A,0.361,0.424,0.215
B,0.279,0.320,0.401
C,0.267,0.349,0.384


## 5. Hitta frekventa itemmängder med FP-growth

**FP-growth** hittar alla kombinationer av items som förekommer i minst `MIN_SUPPORT` av träningstransaktionerna. Den ger samma resultat som *Apriori* men är snabbare, eftersom den inte behöver gå igenom data en gång per itemmängdens storlek.

Bara **träningsdata** används. Teståren hålls helt utanför tills reglerna ska prövas.

In [5]:
frekventa = fpgrowth(items_tran, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LANGD)
print(f"Frekventa itemmängder: {len(frekventa)}")
print(frekventa["itemsets"].map(len).value_counts().sort_index().rename("antal per storlek"))

Frekventa itemmängder: 510
itemsets
1     15
2     81
3    194
4    172
5     48
Name: antal per storlek, dtype: int64


## 6. Skapa regler och behåll de som besvarar frågan

`association_rules` gör alla möjliga regler av de frekventa itemmängderna. De flesta handlar om annat än frågeställningen, till exempel att två konjunkturindikatorer ofta är svaga samtidigt. Sådana regler säger något om konjunkturen, inte om fruktsamhet.

Vi behåller bara regler där **slutledet är en fruktsamhetsklass** och villkoret innehåller **en kommungrupp och minst ett konjunkturvillkor** (se `ar_regeltyp`).

Som kontroll räknas träningsmåtten om med vår egen funktion `matt`. De ska stämma exakt med mlxtends. Då vet vi att samma definitioner används när reglerna prövas på testdata.

In [6]:
alla_regler = association_rules(frekventa, num_itemsets=len(items_tran), metric="confidence", min_threshold=0.0)
regler = alla_regler.loc[[ar_regeltyp(v, s) for v, s in zip(alla_regler["antecedents"], alla_regler["consequents"])]]
regler = pd.DataFrame({
    "villkor": [tuple(sorted(v)) for v in regler["antecedents"]],
    "slutled": [next(iter(s)) for s in regler["consequents"]],
    "confidence_mlxtend": regler["confidence"].to_numpy(),
})

for v, s, c in zip(regler["villkor"], regler["slutled"], regler["confidence_mlxtend"]):
    assert abs(matt(items_tran, ar_tran, v, s)["confidence"] - c) < 1e-12

print(f"Alla regler: {len(alla_regler)}. Regler som besvarar frågan: {len(regler)}")

Alla regler: 5174. Regler som besvarar frågan: 243


## 7. Mät reglerna på tränings- och testår

För varje regel räknas på både tränings- och testdata:

- support, confidence, lift och **antal_ar** (antal olika år bakom villkoret)
- **bas_confidence**: confidence för samma kommungrupp utan konjunkturvillkor
- **tillskott** = confidence − bas_confidence, alltså vad konjunkturen tillför utöver kommuntypen.

Om villkoret aldrig inträffar under testperioden (till exempel en konjunkturklass som inte förekom) går regeln inte att pröva, och testmåtten blir tomma.

In [7]:
def utvardera(regler, items, ar, suffix):
    rader = []
    for villkor, slutled in zip(regler["villkor"], regler["slutled"]):
        m = matt(items, ar, villkor, slutled)
        grupp = next(i for i in villkor if i.startswith("kommungrupp="))
        m["bas_confidence"] = matt(items, ar, (grupp,), slutled)["confidence"]
        m["tillskott"] = m["confidence"] - m["bas_confidence"]
        rader.append({f"{k}_{suffix}": v for k, v in m.items()})
    return pd.DataFrame(rader, index=regler.index)


def utvardera_bada(regler, items_tran, ar_tran, items_test, ar_test):
    return pd.concat([regler[["villkor", "slutled"]],
                      utvardera(regler, items_tran, ar_tran, "tran"),
                      utvardera(regler, items_test, ar_test, "test")], axis=1)


matta = utvardera_bada(regler, items_tran, ar_tran, items_test, ar_test)
print("Mätta regler:", len(matta))

Mätta regler: 243


## 8. Välj ut regler och pröva om de är stabila

En regel **väljs ut** om den på träningsåren uppfyller båda villkoren:
- den bygger på minst `MIN_AR` olika år
- konjunkturvillkoret höjer confidence med minst `MIN_TILLSKOTT` jämfört med bara kommungruppen.

En utvald regel räknas som **stabil** om tillskottet är positivt även på teståren och bygger på minst `MIN_AR_TEST` testår. En regel som bara håller på träningsåren kan vara ett sammanträffande i de år den hittades på.

Kriteriet *stabil* kräver bara att tillskottet är **positivt** på teståren. Titta därför också på **hur stort** det är. Ett testtillskott på några få procentenheter är ett svagt stöd, även om regeln formellt räknas som stabil.

In [8]:
def valj_och_prova(matta):
    valda = matta.loc[(matta["antal_ar_tran"] >= MIN_AR) & (matta["tillskott_tran"] >= MIN_TILLSKOTT)].copy()
    valda["stabil"] = (valda["tillskott_test"] > 0) & (valda["antal_ar_test"] >= MIN_AR_TEST)
    valda["provbar"] = valda["antal_ar_test"] >= MIN_AR_TEST
    return valda.sort_values(["tillskott_tran", "slutled", "villkor"], ascending=[False, True, True])


valda = valj_och_prova(matta)
print(f"Utvalda på träningsåren: {len(valda)}")
print(f"Går att pröva (villkoret förekommer minst {MIN_AR_TEST} testår): {valda['provbar'].sum()}")
print(f"Stabila (positivt tillskott även på teståren): {valda['stabil'].sum()}")

Utvalda på träningsåren: 5
Går att pröva (villkoret förekommer minst 2 testår): 2
Stabila (positivt tillskott även på teståren): 1


## 9. Resultat

Tabellen visar de utvalda reglerna sorterade efter tillskott på träningsåren. Läs den så här:

- **confidence_tran** och **bas_confidence_tran**: hur ofta slutledet gäller med respektive utan konjunkturvillkoret.
- **antal_ar_tran**: hur många olika år regeln vilar på. Få år betyder svagt stöd, även om support ser högt ut.
- **tillskott_test** och **stabil**: om mönstret återkommer under 2016–2025.

In [9]:
visa = valda.assign(villkor=valda["villkor"].map(" & ".join))[[
    "villkor", "slutled", "support_tran", "confidence_tran", "bas_confidence_tran", "tillskott_tran",
    "lift_tran", "antal_ar_tran", "confidence_test", "tillskott_test", "antal_ar_test", "stabil",
]]
pd.set_option("display.max_colwidth", 90)
display(visa.round(3).reset_index(drop=True))

,villkor,slutled,support_tran,confidence_tran,bas_confidence_tran,tillskott_tran,lift_tran,antal_ar_tran,confidence_test,tillskott_test,antal_ar_test,stabil
0,kommungrupp=A & sysselsattning=stark,fruktsamhet=hög,0.017,0.449,0.363,0.086,1.355,3,0.235,0.020,5,True
1,hushallens_konsumtion=svag & kommungrupp=A & sysselsattning=normal,fruktsamhet=låg,0.016,0.321,0.256,0.065,0.949,4,0.239,-0.122,1,False
2,bnp_indikator=svag & hushallens_konsumtion=svag & kommungrupp=A & sysselsattning=normal,fruktsamhet=låg,0.012,0.312,0.256,0.056,0.922,3,0.239,-0.122,1,False
3,bnp_indikator=svag & kommungrupp=A & sysselsattning=normal,fruktsamhet=låg,0.012,0.312,0.256,0.056,0.922,3,0.239,-0.122,1,False
4,bnp_indikator=normal & hushallens_konsumtion=normal & kommungrupp=A,fruktsamhet=hög,0.016,0.413,0.363,0.050,1.246,3,0.207,-0.009,2,False


## 10. Känslighetsanalys

Håller resultatet om vi ändrar de antaganden som är mest osäkra? Samma procedur körs två gånger till:

1. **2 års fördröjning** i stället för 1.
2. **1 års fördröjning utan kommuner med bortfall.** Kommuner med saknade eller interpolerade värden tas bort (flaggan `har_bortfall` från `02`), så att de små kommunernas brus inte styr resultatet.

För varje körning redovisas antalet utvalda och stabila regler och hur många av de stabila reglerna som också är stabila i huvudanalysen.

In [10]:
def kor_arm(underlag):
    items = till_items(underlag)
    tran = underlag["del"] == "traning"
    it_tran, a_tran = items.loc[tran], underlag.loc[tran, "ar"]
    it_test, a_test = items.loc[~tran], underlag.loc[~tran, "ar"]
    frek = fpgrowth(it_tran, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LANGD)
    alla = association_rules(frek, num_itemsets=len(it_tran), metric="confidence", min_threshold=0.0)
    alla = alla.loc[[ar_regeltyp(v, s) for v, s in zip(alla["antecedents"], alla["consequents"])]]
    kandidater = pd.DataFrame({"villkor": [tuple(sorted(v)) for v in alla["antecedents"]],
                               "slutled": [next(iter(s)) for s in alla["consequents"]]})
    return valj_och_prova(utvardera_bada(kandidater, it_tran, a_tran, it_test, a_test))


lag2 = pd.read_csv(PROCESSED / "arm_underlag_lag2.csv", encoding="utf-8-sig", dtype={"Kommunkod": str})
korningar = {
    "lag1 (huvudanalys)": valda,
    "lag2": kor_arm(lag2),
    "lag1 utan bortfall": kor_arm(underlag.loc[~underlag["har_bortfall"]]),
}

stabila_huvud = set(zip(valda.loc[valda["stabil"], "villkor"], valda.loc[valda["stabil"], "slutled"]))
sammanfattning = pd.DataFrame({
    namn: {
        "utvalda": len(df),
        "stabila": int(df["stabil"].sum()),
        "stabila som också är stabila i huvudanalysen": len(
            stabila_huvud & set(zip(df.loc[df["stabil"], "villkor"], df.loc[df["stabil"], "slutled"]))),
    }
    for namn, df in korningar.items()
}).T
display(sammanfattning)

,utvalda,stabila,stabila som också är stabila i huvudanalysen
lag1 (huvudanalys),5,1,1
lag2,7,1,1
lag1 utan bortfall,5,1,1


## 11. Spara

Reglerna från varje körning sparas med tränings- och testmått. Villkoret sparas som text med items åtskilda av ` & `.

In [11]:
filnamn = {"lag1 (huvudanalys)": "arm_regler_lag1.csv", "lag2": "arm_regler_lag2.csv",
           "lag1 utan bortfall": "arm_regler_lag1_utan_bortfall.csv"}
for namn, df in korningar.items():
    utfil = RESULTAT / filnamn[namn]
    df.assign(villkor=df["villkor"].map(" & ".join)).to_csv(utfil, index=False, encoding="utf-8-sig")
    print(f"Sparade {len(df)} regler till {utfil.relative_to(BAS)}")

Sparade 5 regler till resultat/arm_regler_lag1.csv
Sparade 7 regler till resultat/arm_regler_lag2.csv
Sparade 5 regler till resultat/arm_regler_lag1_utan_bortfall.csv


## Att tänka på vid tolkningen

- **Samband, inte orsak.** En regel visar att två saker ofta förekommer tillsammans, inte att den ena orsakar den andra.
- **Få oberoende år.** Konjunkturvillkoren vilar på några få år per klass (se `antal_ar`). Stabila regler är de mest trovärdiga, men även de bygger på ett begränsat antal konjunkturcykler.
- **Många regler prövas.** När många regler prövas samtidigt kommer några att se starka ut av en slump. Kravet på stabilitet under teståren är skyddet mot det, men det är inget formellt signifikanstest.
- **Kommungrupperna gäller 2023.** SKR:s indelning från 2023 används för hela perioden, så grupptillhörigheten beskriver kommunerna som de ser ut i dag.